<a href="https://colab.research.google.com/github/spklapjs/Point-6/blob/main/ai_model/notebooks/01_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json
import pandas as pd
import numpy as np
import torch
from scipy.signal import find_peaks
from google.colab import drive

# 1. 드라이브 마운트
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 디렉토리 경로 설정
BASE_DIR = '/content/drive/MyDrive/point-6/ai_model/data'
RAW_PHONE_DIR = os.path.join(BASE_DIR, 'raw', 'phone')
RAW_SPEN_DIR = os.path.join(BASE_DIR, 'raw', 'spen')
PROCESSED_DIR = os.path.join(BASE_DIR, 'processed')
EXPORT_DIR = '/content/drive/MyDrive/point-6/ai_model/exported_models'

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

# 제약 조건 및 추출 위치 설정 변수 수정 (400ms 윈도우, 330ms 피크, 20프레임 거리 제한)
PHONE_Z_THRESHOLD = 50.0
SPEN_THRESHOLD = 0.5
WINDOW_SIZE = 40
PEAK_POSITION = 33
OVERLAP_DISTANCE = 20

In [3]:
def get_label_idx(label_str):
    label_str = str(label_str)
    if "Cymbal1" in label_str: return 0
    if "Tom1" in label_str: return 1
    if "Cymbal2" in label_str: return 2
    if "Hihat" in label_str or "Hi-hat" in label_str: return 3
    if "Snare" in label_str: return 4
    if "Tom2" in label_str: return 5
    return -1

In [4]:
def process_phone_files():
    features = []
    labels = []
    if not os.path.exists(RAW_PHONE_DIR):
        return np.array(features), np.array(labels)

    for filename in os.listdir(RAW_PHONE_DIR):
        if not filename.endswith('.csv'): continue
        filepath = os.path.join(RAW_PHONE_DIR, filename)
        df = pd.read_csv(filepath)

        if len(df) < WINDOW_SIZE: continue

        z_accel = df['accelZ'].values
        peaks, _ = find_peaks(z_accel, height=PHONE_Z_THRESHOLD, distance=OVERLAP_DISTANCE)

        for peak in peaks:
            start_idx = peak - PEAK_POSITION
            end_idx = start_idx + WINDOW_SIZE

            if start_idx >= 0 and end_idx <= len(df):
                window_df = df.iloc[start_idx:end_idx]

                feature_matrix = np.vstack([
                    window_df['accelX'].values,
                    window_df['accelY'].values,
                    window_df['accelZ'].values,
                    window_df['gyroX'].values,
                    window_df['gyroY'].values,
                    window_df['gyroZ'].values
                ])

                label_idx = get_label_idx(window_df['label'].values.item(0))
                if label_idx != -1:
                    features.append(feature_matrix)
                    labels.append(label_idx)

    return np.array(features), np.array(labels)

def process_spen_files():
    features = []
    labels = []
    if not os.path.exists(RAW_SPEN_DIR):
        return np.array(features), np.array(labels)

    for filename in os.listdir(RAW_SPEN_DIR):
        if not filename.endswith('.csv'): continue
        filepath = os.path.join(RAW_SPEN_DIR, filename)
        df = pd.read_csv(filepath)

        if len(df) < WINDOW_SIZE: continue

        mag = np.maximum(np.abs(df['deltaX'].values), np.abs(df['deltaY'].values))
        peaks, _ = find_peaks(mag, height=SPEN_THRESHOLD, distance=OVERLAP_DISTANCE)

        for peak in peaks:
            start_idx = peak - PEAK_POSITION
            end_idx = start_idx + WINDOW_SIZE

            if start_idx >= 0 and end_idx <= len(df):
                window_df = df.iloc[start_idx:end_idx]

                feature_matrix = np.vstack([
                    window_df['deltaX'].values,
                    window_df['deltaY'].values
                ])

                label_idx = get_label_idx(window_df['label'].values.item(0))
                if label_idx != -1:
                    features.append(feature_matrix)
                    labels.append(label_idx)

    return np.array(features), np.array(labels)

In [5]:
print("스마트폰 데이터 전처리를 시작합니다...")
phone_features, phone_labels = process_phone_files()

if len(phone_features) > 0:
    phone_mean = np.mean(phone_features, axis=(0, 2), keepdims=True)
    phone_std = np.std(phone_features, axis=(0, 2), keepdims=True)
    phone_std[phone_std == 0] = 1e-8

    phone_normalized = (phone_features - phone_mean) / phone_std

    phone_scaling = {
        "mean": phone_mean.flatten().tolist(),
        "std": phone_std.flatten().tolist()
    }
    with open(os.path.join(EXPORT_DIR, 'smartphone_scaling.json'), 'w') as f:
        json.dump(phone_scaling, f)

    torch.save(torch.tensor(phone_normalized, dtype=torch.float32), os.path.join(PROCESSED_DIR, 'smartphone_features.pt'))
    torch.save(torch.tensor(phone_labels, dtype=torch.long), os.path.join(PROCESSED_DIR, 'smartphone_labels.pt'))
    print(f"스마트폰 전처리 완료. 추출된 유효 타격 샘플 수: {len(phone_features)}")
else:
    print("스마트폰 전처리 실패: 유효 타격 샘플이 0개입니다.")

print("에스펜 데이터 전처리를 시작합니다...")
spen_features, spen_labels = process_spen_files()

if len(spen_features) > 0:
    spen_mean = np.mean(spen_features, axis=(0, 2), keepdims=True)
    spen_std = np.std(spen_features, axis=(0, 2), keepdims=True)
    spen_std[spen_std == 0] = 1e-8

    spen_normalized = (spen_features - spen_mean) / spen_std

    spen_scaling = {
        "mean": spen_mean.flatten().tolist(),
        "std": spen_std.flatten().tolist()
    }
    with open(os.path.join(EXPORT_DIR, 'spen_scaling.json'), 'w') as f:
        json.dump(spen_scaling, f)

    torch.save(torch.tensor(spen_normalized, dtype=torch.float32), os.path.join(PROCESSED_DIR, 'spen_features.pt'))
    torch.save(torch.tensor(spen_labels, dtype=torch.long), os.path.join(PROCESSED_DIR, 'spen_labels.pt'))
    print(f"에스펜 전처리 완료. 추출된 유효 타격 샘플 수: {len(spen_features)}")
else:
    print("에스펜 전처리 실패: 유효 타격 샘플이 0개입니다.")

print("모든 데이터 처리 프로세스가 종료되었습니다.")

스마트폰 데이터 전처리를 시작합니다...
스마트폰 전처리 완료. 추출된 유효 타격 샘플 수: 3132
에스펜 데이터 전처리를 시작합니다...
에스펜 전처리 완료. 추출된 유효 타격 샘플 수: 1941
모든 데이터 처리 프로세스가 종료되었습니다.
